In [65]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from datetime import datetime
from tqdm import tqdm
from deepface import DeepFace
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import Normalizer, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

In [66]:
# ==========================================
# 1. CẤU HÌNH HỆ THỐNG LOG
# ==========================================
LOG_DIR = 'log_' + datetime.now().strftime("%Y%m%d_%H%M%S")
if not os.path.exists(LOG_DIR):
    os.makedirs(LOG_DIR)
    print(f"--- Đã tạo thư mục lưu trữ: {LOG_DIR} ---")

# ==========================================
# 2. TIỀN XỬ LÝ & TRÍCH XUẤT ĐẶC TRƯNG
# ==========================================
DATA_PATH = 'D:/facenet/deep_face/deep_face/data/lfw_filtered_10'
MODEL_NAME = "Facenet512"

--- Đã tạo thư mục lưu trữ: log_20260419_153313 ---


In [67]:
def extract_and_save_features():
    x, y = [], []
    folders = sorted(os.listdir(DATA_PATH))
    for folder in tqdm(folders, desc="Extracting Features"):
        folder_path = os.path.join(DATA_PATH, folder)
        for img_name in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_name)
            try:
                embedding_objs = DeepFace.represent(img_path=img_path, model_name=MODEL_NAME, enforce_detection=False)
                x.append(embedding_objs[0]["embedding"])
                y.append(folder)
            except: continue
    return np.array(x), np.array(y)

In [68]:
X, y = extract_and_save_features()
le = LabelEncoder()
y_encoded = le.fit_transform(y)

Extracting Features: 100%|██████████| 158/158 [11:33<00:00,  4.39s/it]


In [69]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

In [70]:
l2_normalizer = Normalizer(norm='l2')
X_train = l2_normalizer.transform(X_train)
X_test = l2_normalizer.transform(X_test)

In [71]:
model = Sequential([
    Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.4),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dense(len(le.classes_), activation='softmax')
])

In [72]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [73]:
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), 
                    epochs=100, batch_size=32, callbacks=[early_stop], verbose=1)

Epoch 1/100
109/109 [==============================] - 1s 5ms/step - loss: 2.0656 - accuracy: 0.6317 - val_loss: 4.4317 - val_accuracy: 0.2798
Epoch 2/100
109/109 [==============================] - 0s 4ms/step - loss: 0.7198 - accuracy: 0.8896 - val_loss: 3.6602 - val_accuracy: 0.4786
Epoch 3/100
109/109 [==============================] - 0s 4ms/step - loss: 0.4961 - accuracy: 0.9202 - val_loss: 2.4923 - val_accuracy: 0.7699
Epoch 4/100
109/109 [==============================] - 0s 4ms/step - loss: 0.3558 - accuracy: 0.9335 - val_loss: 1.3916 - val_accuracy: 0.8844
Epoch 5/100
109/109 [==============================] - 0s 4ms/step - loss: 0.2713 - accuracy: 0.9506 - val_loss: 0.6980 - val_accuracy: 0.8936
Epoch 6/100
109/109 [==============================] - 0s 4ms/step - loss: 0.2470 - accuracy: 0.9485 - val_loss: 0.5360 - val_accuracy: 0.9029
Epoch 7/100
109/109 [==============================] - 0s 4ms/step - loss: 0.1802 - accuracy: 0.9668 - val_loss: 0.5324 - val_accuracy: 0.9098

In [74]:
# Lưu model
model.save(os.path.join(LOG_DIR, 'face512_model.h5'))

In [75]:
# ==========================================
# 4. LƯU BIỂU ĐỒ VÀ KẾT QUẢ (LOGGING)
# ==========================================

# 4.1. Lưu biểu đồ Training History
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()
plt.savefig(os.path.join(LOG_DIR, 'training_history.png'))
plt.close()

# 4.2. Lưu Confusion Matrix (Heatmap)
y_pred = np.argmax(model.predict(X_test), axis=1)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(15, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'confusion_matrix.png'))
plt.close()

# 4.3. Tính toán và lưu biểu đồ FAR/FRR/EER
y_probs = model.predict(X_test)
y_true_probs = np.max(y_probs, axis=1)
thresholds = np.linspace(0, 1, 100)
far_list, frr_list = [], []

28/28 [==============================] - 0s 1ms/step


In [76]:
for t in thresholds:
    tp = np.sum((y_true_probs >= t) & (y_pred == y_test))
    fp = np.sum((y_true_probs >= t) & (y_pred != y_test))
    fn = np.sum((y_true_probs < t) & (y_pred == y_test))
    tn = np.sum((y_true_probs < t) & (y_pred != y_test))
    far_list.append(fp / (fp + tn) if (fp + tn) > 0 else 0)
    frr_list.append(fn / (fn + tp) if (fn + tp) > 0 else 0)

eer_idx = np.argmin(np.abs(np.array(far_list) - np.array(frr_list)))
eer = far_list[eer_idx]

plt.figure(figsize=(8, 6))
plt.plot(thresholds, far_list, label='FAR')
plt.plot(thresholds, frr_list, label='FRR')
plt.axvline(thresholds[eer_idx], color='r', linestyle='--', label=f'EER: {eer:.4f}')
plt.title('FAR vs FRR')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(LOG_DIR, 'eer_analysis.png'))
plt.close()

In [77]:
# 4.4. Xuất file văn bản tổng hợp kết quả (Report)
report = classification_report(y_test, y_pred, target_names=le.classes_)
with open(os.path.join(LOG_DIR, 'final_report.txt'), 'w', encoding='utf-8') as f:
    f.write("--- BÁO CÁO KẾT QUẢ NHẬN DIỆN ---\n")
    f.write(f"Thời gian: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Model trích xuất: {MODEL_NAME}\n")
    f.write(f"EER đạt được: {eer:.4f} tại Threshold: {thresholds[eer_idx]:.4f}\n\n")
    f.write("--- CHI TIẾT CLASSIFICATION REPORT ---\n")
    f.write(report)

print(f"--- TẤT CẢ KẾT QUẢ ĐÃ ĐƯỢC LƯU TẠI: {LOG_DIR} ---")

--- TẤT CẢ KẾT QUẢ ĐÃ ĐƯỢC LƯU TẠI: log_20260419_153313 ---
